# V2 Spec & Registry: Declarative Configuration

BaseAttentive v2.2.0 uses a **spec / registry / resolver** workflow
that decouples model *description* from model *construction*. This
notebook walks through:

1. `BaseAttentiveSpec` — backend-neutral model configuration
2. JSON serialisation and reload
3. `BaseAttentiveComponentSpec` — selecting component keys
4. `ComponentRegistry` — registering a custom builder
5. `assemble_model()` — resolving a `BaseAttentiveV2Assembly`
6. `BaseAttentiveV2` — building a trainable model from the spec

## Why declarative configuration?

- **Version-control** model configs as plain JSON
- **Reproduce** experiments exactly from a saved spec
- **Swap backends** (`tensorflow` / `torch` / `jax`) without changing model code
- **Override components** through registry keys instead of constructor rewrites

## Setup

In [ ]:
# ── v2.2.0 Backend Setup ─────────────────────────────────────────────────────
# BASE_ATTENTIVE_BACKEND must be set *before* importing base_attentive.
# Choose your installed backend: "tensorflow" | "torch" | "jax" | "auto"
import os
os.environ.setdefault("BASE_ATTENTIVE_BACKEND", "tensorflow")
os.environ.setdefault("KERAS_BACKEND", os.environ["BASE_ATTENTIVE_BACKEND"])
BACKEND = os.environ["BASE_ATTENTIVE_BACKEND"]
print(f"Backend: {BACKEND}")

In [ ]:
import json
import pathlib
from dataclasses import replace

import numpy as np

from base_attentive import BaseAttentive, __version__
from base_attentive.config import (
    BaseAttentiveArchitectureSpec,
    BaseAttentiveComponentSpec,
    BaseAttentiveRuntimeSpec,
    BaseAttentiveSpec,
    normalize_base_attentive_spec,
    serialize_base_attentive_spec,
)
from base_attentive.experimental import BaseAttentiveV2
from base_attentive.registry import (
    ComponentRegistry,
    ModelRegistry,
)
from base_attentive.resolver import (
    BackendContext,
    BaseAttentiveV2Assembly,
    assemble_model,
    ensure_backend_registrations,
)

component_registry = ComponentRegistry()
model_registry = ModelRegistry()
backend_context = BackendContext.current(BACKEND)
component_registry, model_registry = (
    ensure_backend_registrations(
        backend_context=backend_context,
        component_registry=component_registry,
        model_registry=model_registry,
    )
)

print(f"BaseAttentive {__version__} spec imports OK")
print("Resolver backend:", backend_context.name)

---

## 1 — Keyword approach (recap)

The classic keyword approach is fine for quick experiments.  Everything
shown here also applies to v1-style construction.

In [ ]:
model_kw = BaseAttentive(
    static_input_dim=4,
    dynamic_input_dim=8,
    future_input_dim=6,
    output_dim=2,
    forecast_horizon=24,
    embed_dim=64,
    num_heads=8,
    dropout_rate=0.15,
)
print("Keyword model:", model_kw.name)

---

## 2 — `BaseAttentiveSpec`

`BaseAttentiveSpec` is a frozen dataclass that stores the full model
configuration. The current schema groups fields as follows:

| Group | Fields |
|-------|--------|
| **Required I/O** | `static_input_dim`, `dynamic_input_dim`, `future_input_dim`, `output_dim`, `forecast_horizon` |
| **Capacity** | `embed_dim`, `hidden_units`, `attention_heads`, `dropout_rate`, `activation` |
| **Architecture** | `architecture.encoder_type`, `architecture.decoder_attention_stack`, `architecture.feature_processing` |
| **Runtime** | `runtime.num_encoder_layers`, `runtime.scales`, `runtime.multi_scale_agg`, `runtime.memory_size`, `runtime.use_residuals` |
| **Output** | `head_type`, `quantiles` |
| **Backend** | `backend_name` |
| **Component keys** | `components.*` |


In [ ]:
spec = BaseAttentiveSpec(
    static_input_dim=4,
    dynamic_input_dim=8,
    future_input_dim=6,
    output_dim=2,
    forecast_horizon=24,
    embed_dim=64,
    hidden_units=96,
    attention_heads=8,
    dropout_rate=0.15,
    head_type="quantile",
    quantiles=(0.1, 0.5, 0.9),
    backend_name="tensorflow",
    architecture=BaseAttentiveArchitectureSpec(
        encoder_type="hybrid",
        decoder_attention_stack=("cross", "hierarchical"),
        feature_processing="dense",
    ),
    runtime=BaseAttentiveRuntimeSpec(
        num_encoder_layers=2,
        memory_size=64,
        multi_scale_agg="last",
    ),
)

print("spec.embed_dim      :", spec.embed_dim)
print("spec.quantiles      :", spec.quantiles)
print("spec.backend_name   :", spec.backend_name)
print("spec.objective      :", spec.objective)
print("spec.attention_lvls :", spec.attention_levels)

### Specs are immutable

Specs are frozen dataclasses — attempting to mutate one raises a
`FrozenInstanceError`.  To "modify" a spec, use `dataclasses.replace`:

In [ ]:
spec_torch = replace(
    spec, backend_name="torch", dropout_rate=0.2
)

print(
    "Original backend:",
    spec.backend_name,
    "dropout:",
    spec.dropout_rate,
)
print(
    "Modified backend:",
    spec_torch.backend_name,
    "dropout:",
    spec_torch.dropout_rate,
)

---

## 3 — JSON Serialisation

Specs serialise to/from plain JSON — no pickles, no framework artefacts.

In [ ]:
# Serialise to dict -> JSON string
spec_dict = serialize_base_attentive_spec(spec)
spec_json = json.dumps(spec_dict, indent=2)

print(spec_json[:600], "...")

In [ ]:
# Reload from JSON
reloaded_dict = json.loads(spec_json)
spec_reload = normalize_base_attentive_spec(reloaded_dict)

assert spec_reload.embed_dim == spec.embed_dim
assert spec_reload.quantiles == spec.quantiles
print("Spec round-trip: OK")
print("Reloaded backend:", spec_reload.backend_name)

In [ ]:
# Save to file
spec_path = pathlib.Path("model_spec.json")
spec_path.write_text(spec_json)
print(f"Spec saved to: {spec_path.resolve()}")

# Load from file
spec_from_file = normalize_base_attentive_spec(
    json.loads(spec_path.read_text())
)
print(
    "Loaded from file — embed_dim:", spec_from_file.embed_dim
)

---

## 4 — `BaseAttentiveComponentSpec`

`BaseAttentiveComponentSpec` stores the registry keys used for each
logical part of the model. Instead of passing free-form per-component
parameter dictionaries, the current API selects named builders such as
`"encoder.temporal_self_attention"` or `"pool.mean"`.


In [ ]:
component_keys = BaseAttentiveComponentSpec(
    dynamic_encoder="encoder.temporal_self_attention",
    future_encoder="encoder.temporal_self_attention",
    sequence_pooling="pool.mean",
    fusion="fusion.concat",
    quantile_head="head.quantile_forecast",
)

spec_with_components = replace(
    spec, components=component_keys
)

print(
    "Dynamic encoder key:",
    spec_with_components.components.dynamic_encoder,
)
print(
    "Sequence pool key :",
    spec_with_components.components.sequence_pooling,
)
print(
    "Quantile head key :",
    spec_with_components.components.quantile_head,
)

---

## 5 — `ComponentRegistry`: Registering a Custom Builder

The registry stores builder callables keyed by a backend-neutral string.
Below, we register a tiny demo pooling builder into a local registry.


In [ ]:
def build_demo_last_pool(
    *,
    context=None,
    spec=None,
    axis=1,
    keepdims=False,
    **kwargs,
):
    def _pool(x, training=False):
        return x[:, -1:, :] if keepdims else x[:, -1, :]

    return _pool


print("Custom builder defined")

In [ ]:
## Use the custom component in a spec
spec_custom = replace(
    spec_with_components,
    components=replace(
        spec_with_components.components,
        sequence_pooling="pool.demo_last",
    ),
)

assembly = assemble_model(
    "base_attentive.v2",
    spec=spec_custom,
    backend_context=backend_context,
    component_registry=component_registry,
    model_registry=model_registry,
)

print("Assembly type       :", type(assembly).__name__)
print(
    "Custom pool in spec :",
    spec_custom.components.sequence_pooling,
)
print(
    "Resolved pool type  :",
    type(assembly.sequence_pool).__name__,
)

In [ ]:
REGISTRY_KEY = "encoder.residual_bilstm"


def build_residual_bilstm(
    *,
    context=None,
    spec=None,
    units=64,
    num_layers=1,
    name=None,
    **kwargs,
):
    layers_ns = (
        context.layers
        if context is not None
        and getattr(context, "layers", None) is not None
        else BackendContext.current("tensorflow").layers
    )
    dense_cls = layers_ns.Dense
    layer_cls = getattr(layers_ns, "Layer", object)

    class ResidualBiLSTM(layer_cls):
        def __init__(self, units, num_layers, name=None):
            try:
                super().__init__(name=name)
            except TypeError:
                super().__init__()
                if name is not None:
                    self.name = name
            self.units = units
            self.num_layers = num_layers
            self.proj = dense_cls(
                units, name=f"{name}_proj" if name else None
            )

        def call(self, inputs, training=False):
            projected = self.proj(inputs)
            input_shape = getattr(inputs, "shape", None)
            output_shape = getattr(projected, "shape", None)
            if input_shape == output_shape:
                return projected + inputs
            return projected

        def get_config(self):
            base_config = {}
            parent = getattr(super(), "get_config", None)
            if callable(parent):
                base_config = dict(parent())
            base_config.update(
                {
                    "units": self.units,
                    "num_layers": self.num_layers,
                }
            )
            return base_config

    return ResidualBiLSTM(
        units=units,
        num_layers=num_layers,
        name=name,
    )


component_registry.register(
    key=REGISTRY_KEY,
    builder=build_residual_bilstm,
    backend="generic",
    description="Bidirectional LSTM-style demo encoder with residual projection",
    replace=True,
)

print("Registered encoder keys:")
for key in component_registry.list_keys():
    if key.startswith("encoder."):
        print(" ", key)

In [ ]:
# Test the builder directly
registration = component_registry.resolve(
    REGISTRY_KEY,
    backend=backend_context.name,
    allow_generic=True,
)
layer = registration.builder(
    context=backend_context,
    spec=spec_with_components,
    units=128,
    num_layers=3,
    name="test_enc",
)
print("Resolved layer type:", type(layer).__name__)
print("Layer units:", layer.units)
print("Layer depth:", layer.num_layers)

### Use the custom encoder in a spec

In [ ]:
spec_custom = replace(
    spec,
    components=replace(
        spec.components,
        dynamic_encoder=REGISTRY_KEY,
        future_encoder=REGISTRY_KEY,
    ),
)

print(
    "Custom encoder component type:",
    spec_custom.components.dynamic_encoder,
)

---

## 6 — Building `BaseAttentiveV2` from a Spec

The resolved `BaseAttentiveV2Assembly` is a low-level view of the
components. For normal use, instantiate `BaseAttentiveV2` with the same
spec and let it manage the assembly internally.


In [ ]:
model_from_spec = BaseAttentiveV2(
    static_input_dim=spec_custom.static_input_dim,
    dynamic_input_dim=spec_custom.dynamic_input_dim,
    future_input_dim=spec_custom.future_input_dim,
    output_dim=spec_custom.output_dim,
    forecast_horizon=spec_custom.forecast_horizon,
    spec=spec_custom,
    backend_name=spec_custom.backend_name,
    name="spec_driven_v2",
)

print("Model type      :", type(model_from_spec).__name__)
print("Forecast horizon:", model_from_spec.forecast_horizon)
print("Head type       :", model_from_spec.spec.head_type)

In [ ]:
# Quick smoke-test: forward pass
B, T, H = 8, 24, 24
x_s = np.random.randn(B, 4).astype("float32")
x_d = np.random.randn(B, T, 8).astype("float32")
x_f = np.random.randn(B, H, 6).astype("float32")

out = model_from_spec([x_s, x_d, x_f])
print("Output shape:", np.array(out).shape)

---

## 7 — Inspecting the Assembly

The assembly object exposes the concrete components chosen by the
resolver. This is useful for debugging, backend inspection, and custom
pipelines.


In [ ]:
print("Assembled model name  :", model_from_spec.name)
print("Embed dim from spec   :", spec_custom.embed_dim)
print(
    "Encoder key in spec   :",
    spec_custom.components.dynamic_encoder,
)
print(
    "Assembly dataclass    :",
    BaseAttentiveV2Assembly.__name__,
)
print(
    "Sequence pool object  :",
    type(assembly.sequence_pool).__name__,
)

In [ ]:
print(
    "Hidden projection:",
    type(assembly.hidden_projection).__name__,
)
print(
    "Output head      :", type(assembly.output_head).__name__
)
print("Backend context  :", assembly.backend_context.name)

---

## 8 — Sharing Specs Across Experiments

A common pattern is to define a **base spec** and derive per-experiment
variants with `dataclasses.replace`.


In [ ]:
BASE_SPEC = BaseAttentiveSpec(
    static_input_dim=4,
    dynamic_input_dim=8,
    future_input_dim=6,
    output_dim=2,
    forecast_horizon=24,
    embed_dim=64,
    attention_heads=8,
    dropout_rate=0.1,
)

# Experiment A: TensorFlow, quantile output
spec_A = replace(
    BASE_SPEC,
    backend_name="tensorflow",
    head_type="quantile",
    quantiles=(0.1, 0.5, 0.9),
)

# Experiment B: PyTorch, point output, bigger capacity
spec_B = replace(
    BASE_SPEC,
    backend_name="torch",
    embed_dim=128,
    hidden_units=128,
    dropout_rate=0.2,
)

# Experiment C: JAX, quantile output
spec_C = replace(
    BASE_SPEC,
    backend_name="jax",
    head_type="quantile",
    quantiles=(0.2, 0.5, 0.8),
)

experiment_specs = {"A": spec_A, "B": spec_B, "C": spec_C}
for name, s in experiment_specs.items():
    pathlib.Path(f"spec_exp_{name}.json").write_text(
        json.dumps(serialize_base_attentive_spec(s), indent=2)
    )
    print(
        f"spec_exp_{name}.json saved — backend={s.backend_name}, "
        f"head_type={s.head_type}"
    )

---

## Summary

| Concept | Purpose |
|---------|--------|
| `BaseAttentiveSpec` | Frozen, JSON-serialisable model blueprint |
| `BaseAttentiveComponentSpec` | Declarative selection of component keys |
| `ComponentRegistry.register()` | Plug in a custom builder by string key |
| `assemble_model()` | Resolve a `BaseAttentiveV2Assembly` from a spec |
| `BaseAttentiveV2` | Trainable resolver-driven model scaffold |
| `serialize_base_attentive_spec()` | Stable JSON export for saved experiments |

## Next Steps

- [06_crps_probabilistic_forecasting.ipynb](06_crps_probabilistic_forecasting.ipynb) — probabilistic heads and CRPS workflows
- [Documentation](https://base-attentive.readthedocs.io/en/latest/) — API and usage guides
